In [ ]:
# import data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# change this to ur directory!!!
GTFS_DIR = '/content/drive/MyDrive/Transit Data Challenge/Dataset/Complete GTFS'

import os, zipfile, re
import pandas as pd
import geopandas as gpd
import folium
from IPython.display import display

In [ ]:
def read_gtfs(filename, **kwargs):
    return pd.read_csv(os.path.join(GTFS_DIR, filename), **kwargs)

routes     = read_gtfs('routes.txt')
trips      = read_gtfs('trips.txt', low_memory=False)
stops      = read_gtfs('stops.txt')
shapes_raw = read_gtfs('shapes.txt', dtype=str)

# route IDs: 1 = yonge-university, 2 = bloor-danforth, 4 = sheppard
SUBWAY_ROUTE_IDS = [1, 2, 4]
subway_trips     = trips[trips['route_id'].isin(SUBWAY_ROUTE_IDS)].copy()
subway_trip_ids  = set(subway_trips['trip_id'].astype(str))

# stream stop_times - it's large so chunk it
chunks = []
for chunk in pd.read_csv(os.path.join(GTFS_DIR, 'stop_times.txt'), chunksize=200_000, dtype=str):
    filtered = chunk[chunk['trip_id'].isin(subway_trip_ids)]
    if len(filtered):
        chunks.append(filtered)
subway_stop_times = pd.concat(chunks, ignore_index=True)

In [ ]:
# build station list
def clean_station_name(raw):
    name = re.sub(r'\s*-\s*(Northbound|Southbound|Eastbound|Westbound).*', '', raw)
    name = re.sub(r'\s*Platform.*', '', name)
    name = re.sub(r'\s*-\s*Subway.*', '', name)
    return name.strip()

subway_trips['trip_id'] = subway_trips['trip_id'].astype(str)
stops['stop_id']        = stops['stop_id'].astype(str)

st = (subway_stop_times
      .merge(subway_trips[['trip_id', 'route_id']], on='trip_id', how='left')
      [['route_id', 'stop_id']]
      .drop_duplicates()
      .merge(stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='left'))

st['station_name'] = st['stop_name'].apply(clean_station_name)
st['stop_lat']     = st['stop_lat'].astype(float)
st['stop_lon']     = st['stop_lon'].astype(float)

stations = st.drop_duplicates(subset=['route_id', 'station_name']).reset_index(drop=True)

In [ ]:
# build polylines (lines 1, 2, 4 from gtfs shapes)
shapes_raw['shape_pt_lat']      = shapes_raw['shape_pt_lat'].astype(float)
shapes_raw['shape_pt_lon']      = shapes_raw['shape_pt_lon'].astype(float)
shapes_raw['shape_pt_sequence'] = shapes_raw['shape_pt_sequence'].astype(int)

def get_longest_shape(route_id):
    shape_ids = subway_trips[subway_trips['route_id'] == route_id]['shape_id'].unique()
    return max(shape_ids, key=lambda sid: len(shapes_raw[shapes_raw['shape_id'] == sid]))

route_polylines = {}
for route_id in [1, 2, 4]:
    shape_id = get_longest_shape(route_id)
    pts = shapes_raw[shapes_raw['shape_id'] == shape_id].sort_values('shape_pt_sequence')
    route_polylines[route_id] = list(zip(pts['shape_pt_lat'], pts['shape_pt_lon']))

#  add line 3 from shapefile (not in current gtfs - closed july 2023)
# source: google maps
LINE3_STATIONS = [
    {'name': 'Kennedy Station',            'lat': 43.73205491519861, 'lon': -79.26473171879161},
    {'name': 'Lawrence East Station',      'lat': 43.75090042148507, 'lon': -79.27044776112001},
    {'name': 'Ellesmere Station',          'lat': 43.76708403355233, 'lon': -79.27742831637770},
    {'name': 'Midland Station',            'lat': 43.77059611255155, 'lon': -79.27214037830714},
    {'name': 'Scarborough Centre Station', 'lat': 43.77450825133060, 'lon': -79.25791334941826},
    {'name': 'McCowan Station',            'lat': 43.77513248991619, 'lon': -79.25183956111863},
]

ZIP_PATH    = '/content/drive/MyDrive/Transit Data Challenge/Dataset/ttc-subway-shapefile-wgs84.zip' # alsooooo change to ur directory
EXTRACT_DIR = '/tmp/ttc_subway_shp'

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)

shp_path = next(os.path.join(EXTRACT_DIR, f) for f in os.listdir(EXTRACT_DIR) if f.endswith('.shp'))
gdf = gpd.read_file(shp_path)

for _, row in gdf.iterrows():
    rid = int(row['RID'])
    if rid == 3:
        route_polylines[3] = [(s['lat'], s['lon']) for s in LINE3_STATIONS]


In [ ]:
LINE_CONFIG = {
    1: {'name': 'Line 1 · Yonge-University', 'color': '#FFCD00', 'weight': 5, 'dash': None},
    2: {'name': 'Line 2 · Bloor-Danforth',   'color': '#00A650', 'weight': 5, 'dash': None},
    3: {'name': 'Line 3 · Scarborough RT',   'color': '#0054A6', 'weight': 4, 'dash': None},
    4: {'name': 'Line 4 · Sheppard',         'color': '#B000B0', 'weight': 5, 'dash': None},
}

m = folium.Map(location=[43.700, -79.420], zoom_start=12, tiles='CartoDB positron', control_scale=True)

for route_id, cfg in LINE_CONFIG.items():
    if route_id not in route_polylines:
        continue

    layer = folium.FeatureGroup(name=cfg['name'], show=True)

    polyline_kwargs = dict(
        locations=route_polylines[route_id],
        color=cfg['color'],
        weight=cfg['weight'],
        opacity=0.75 if route_id == 3 else 0.85,
        tooltip=cfg['name'] + (' (Closed 2023)' if route_id == 3 else ''),
    )
    if cfg['dash']:
        polyline_kwargs['dash_array'] = cfg['dash']
    folium.PolyLine(**polyline_kwargs).add_to(layer)

    # station markers
    if route_id == 3:
        for s in LINE3_STATIONS:
            folium.CircleMarker(
                location=[s['lat'], s['lon']],
                radius=6, color='white', weight=2,
                fill=True, fill_color='#0054A6', fill_opacity=1.0,
                tooltip=folium.Tooltip(f"<b>{s['name']}</b><br/>Line 3 · Scarborough RT (Closed 2023)", sticky=False),
                popup=folium.Popup(f"<b>{s['name']}</b><br/>Line 3 · Scarborough RT<br/><small>({s['lat']}, {s['lon']})</small>", max_width=220),
            ).add_to(layer)
    else:
        for _, row in stations[stations['route_id'] == route_id].iterrows():
            folium.CircleMarker(
                location=[row['stop_lat'], row['stop_lon']],
                radius=6, color='white', weight=2,
                fill=True, fill_color=cfg['color'], fill_opacity=1.0,
                tooltip=folium.Tooltip(f"<b>{row['station_name']}</b><br/>{cfg['name']}", sticky=False),
                popup=folium.Popup(f"<b>{row['station_name']}</b><br/>{cfg['name']}<br/><small>({row['stop_lat']:.5f}, {row['stop_lon']:.5f})</small>", max_width=220),
            ).add_to(layer)

    layer.add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;background:white;border:1px solid #ccc;
border-radius:8px;padding:12px 16px;font-family:Arial,sans-serif;font-size:13px;
z-index:9999;box-shadow:2px 2px 6px rgba(0,0,0,0.2)">
  <b style='font-size:14px'>TTC Subway</b><br/><br/>
  <span style='color:#FFCD00'>&#9644;</span> Line 1 · Yonge-University<br/>
  <span style='color:#00A650'>&#9644;</span> Line 2 · Bloor-Danforth<br/>
  <span style='color:#0054A6'>&#9644;</span> Line 3 · Scarborough RT <i>(Closed 2023)</i><br/>
  <span style='color:#B000B0'>&#9644;</span> Line 4 · Sheppard<br/>
  <br/><small>hover station for name · click for details</small>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)

display(m)